In [ ]:
from google.colab import drive
import pandas as pd

# 구글 드라이브 마운트
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 분석할 파일 경로 목록
file_paths = [
    '/content/drive/MyDrive/09.개인_CB정보/201912_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202012_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202112_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202212_개인CB.csv'
]

# 씬파일러 판별에 사용할 컬럼들
target_cols = ['C1M210000', 'C18233003', 'C18233004', 'C18233005', 'L10220000']

results = []

for path in file_paths:
    # 파일 이름에서 연도 추출 (예: '201912')
    year_month = path.split('/')[-1].split('_')[0]
    print(f"{year_month} 데이터 처리 중...")

    try:
        # 메모리 절약을 위해 필요한 컬럼만 로드
        df = pd.read_csv(path, usecols=target_cols)

        # 5개 컬럼의 값이 모두 0인 사람을 씬파일러로 정의
        is_thin_filer = (
            (df['C1M210000'] == 0) &
            (df['C18233003'] == 0) &
            (df['C18233004'] == 0) &
            (df['C18233005'] == 0) &
            (df['L10220000'] == 0)
        )

        thin_filer_count = is_thin_filer.sum()
        total_count = len(df)
        ratio = (thin_filer_count / total_count) * 100 if total_count > 0 else 0

        results.append({
            '기준년월': year_month,
            '전체 고객 수': total_count,
            '씬파일러 수': thin_filer_count,
            '씬파일러 비중(%)': round(ratio, 2)
        })

    except Exception as e:
        print(f"{year_month} 데이터 처리 중 오류 발생: {e}")

# 결과를 데이터프레임으로 변환하여 출력
result_df = pd.DataFrame(results)
display(result_df)

201912 데이터 처리 중...
202012 데이터 처리 중...
202112 데이터 처리 중...
202212 데이터 처리 중...


,기준년월,전체 고객 수,씬파일러 수,씬파일러 비중(%)
0,201912,3129036,802882,25.66
1,202012,3129036,823170,26.31
2,202112,3129036,833405,26.63
3,202212,3129036,794773,25.40


In [ ]:
import pandas as pd

# 분석할 파일 경로 목록
file_paths = [
    '/content/drive/MyDrive/09.개인_CB정보/201912_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202012_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202112_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202212_개인CB.csv'
]

# 분석 대상 컬럼 및 한글명 매핑
target_cols = ['AL012G005', 'AL012G011', 'AL012G019']
col_names_kor = {
    'AL012G005': '3년내 자택주소 이력건수(구간화)',
    'AL012G011': '3년내 직장명 이력건수(구간화)',
    'AL012G019': '3년내 휴대폰번호 이력건수(구간화)'
}

# 결과를 저장할 딕셔너리 초기화
results_dict = {col: {'missing': {}, 'dist': {}} for col in target_cols}
years = []

print("데이터를 분석 중입니다. 잠시만 기다려주세요...")
for path in file_paths:
    year_month = path.split('/')[-1].split('_')[0]
    years.append(year_month)

    try:
        # 메모리 절약을 위해 필요한 컬럼만 로드
        df = pd.read_csv(path, usecols=target_cols)
        total_rows = len(df)

        for col in target_cols:
            # 결측치 계산 및 저장
            missing_count = df[col].isnull().sum()
            missing_ratio = (missing_count / total_rows) * 100
            results_dict[col]['missing'][year_month] = f"{missing_count:,}개 ({missing_ratio:.2f}%)"

            # 데이터 분포 계산 및 저장
            value_counts = df[col].value_counts(dropna=False).sort_index()
            value_ratios = df[col].value_counts(dropna=False, normalize=True).sort_index() * 100

            dist_df = pd.DataFrame({
                '개수': value_counts,
                '비율(%)': value_ratios.round(2)
            })
            results_dict[col]['dist'][year_month] = dist_df

    except Exception as e:
        print(f"{year_month} 데이터 처리 중 오류 발생: {e}")

print("\n분석 완료! 결과를 출력합니다.\n")

# 컬럼별로 결과 묶어서 출력
for col in target_cols:
    kor_name = col_names_kor[col]
    print(f"\n{'='*80}")
    print(f"▶ {col} ({kor_name})")
    print(f"{'='*80}")

    # 결측치 현황 출력
    print("\n[결측치 현황]")
    for year in years:
        if year in results_dict[col]['missing']:
            print(f"- {year}: {results_dict[col]['missing'][year]}")

    # 데이터 분포 현황 병합 및 출력
    print("\n[데이터 분포 현황]")
    dist_dfs = []
    valid_years = []
    for year in years:
        if year in results_dict[col]['dist']:
            dist_dfs.append(results_dict[col]['dist'][year])
            valid_years.append(year)

    if dist_dfs:
        # 다중 인덱스(MultiIndex) 컬럼을 생성하여 연도별로 예쁘게 출력
        combined_df = pd.concat(dist_dfs, axis=1, keys=valid_years)
        combined_df.index.name = '값'
        combined_df = combined_df.fillna(0) # 특정 연도에만 없는 값은 0으로 표시
        display(combined_df)


데이터를 분석 중입니다. 잠시만 기다려주세요...

분석 완료! 결과를 출력합니다.


▶ AL012G005 (3년내 자택주소 이력건수(구간화))

[결측치 현황]
- 201912: 0개 (0.00%)
- 202012: 0개 (0.00%)
- 202112: 0개 (0.00%)
- 202212: 0개 (0.00%)

[데이터 분포 현황]


201912          202012          202112          202212       
         개수  비율(%)       개수  비율(%)       개수  비율(%)       개수  비율(%)
값                                                                 
-9       51   0.00       41   0.00       37   0.00       33   0.00
 1   780095  24.93   867896  27.74   951444  30.41   875077  27.97
 2  1391820  44.48  1441433  46.07  1670398  53.38  1842223  58.88
 3   615571  19.67   530980  16.97   325180  10.39   269516   8.61
 4   226370   7.23   172374   5.51    78962   2.52    55083   1.76
 5    79050   2.53    79726   2.55    70856   2.26    59938   1.92
 6    25078   0.80    25542   0.82    22327   0.71    18976   0.61
 7    11001   0.35    11044   0.35     9832   0.31     8190   0.26


▶ AL012G011 (3년내 직장명 이력건수(구간화))

[결측치 현황]
- 201912: 0개 (0.00%)
- 202012: 0개 (0.00%)
- 202112: 0개 (0.00%)
- 202212: 0개 (0.00%)

[데이터 분포 현황]


201912          202012          202112          202212       
         개수  비율(%)       개수  비율(%)       개수  비율(%)       개수  비율(%)
값                                                                 
-9       50   0.00       42   0.00       42   0.00       30   0.00
 1  2106310  67.31  2133096  68.17  2190368  70.00  2217708  70.88
 2   801030  25.60   776865  24.83   747887  23.90   744736  23.80
 3   180651   5.77   178589   5.71   155488   4.97   135856   4.34
 4    33568   1.07    33129   1.06    28923   0.92    25106   0.80
 5     7427   0.24     7315   0.23     6328   0.20     5600   0.18


▶ AL012G019 (3년내 휴대폰번호 이력건수(구간화))

[결측치 현황]
- 201912: 0개 (0.00%)
- 202012: 0개 (0.00%)
- 202112: 0개 (0.00%)
- 202212: 0개 (0.00%)

[데이터 분포 현황]


201912          202012          202112          202212       
         개수  비율(%)       개수  비율(%)       개수  비율(%)       개수  비율(%)
값                                                                 
-9       42   0.00       43   0.00       40   0.00       33   0.00
 1   943633  30.16  1003478  32.07  1128235  36.06  1128361  36.06
 2  1924927  61.52  1871118  59.80  1783335  56.99  1815593  58.02
 3   218194   6.97   213371   6.82   182451   5.83   155164   4.96
 4    31348   1.00    30429   0.97    25733   0.82    22208   0.71
 5    10892   0.35    10597   0.34     9242   0.30     7677   0.25

In [ ]:
import pandas as pd

# 분석할 파일 경로 목록
file_paths = [
    '/content/drive/MyDrive/09.개인_CB정보/201912_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202012_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202112_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202212_개인CB.csv'
]

# 분석 대상 6개 컬럼 및 한글명 매핑
target_cols = ['L10210800', 'L10210B00', 'L90210300', 'L90210200', 'L10210M00', 'L10210000']
col_names_kor = {
    'L10210800': '저축은행대출건수',
    'L10210B00': '보험대출건수',
    'L90210300': '할부금융대출건수',
    'L90210200': '카드대출건수',
    'L10210M00': '기타업종대출건수',
    'L10210000': '총대출건수'
}

# 결과를 저장할 딕셔너리 초기화
results_dict = {col: {'missing': {}, 'dist': {}} for col in target_cols}
years = []

print("대출 관련 6개 컬럼 데이터를 분석 중입니다. 잠시만 기다려주세요...")
for path in file_paths:
    year_month = path.split('/')[-1].split('_')[0]
    years.append(year_month)

    try:
        # 메모리 절약을 위해 필요한 컬럼만 로드
        df = pd.read_csv(path, usecols=target_cols)
        total_rows = len(df)

        for col in target_cols:
            # 결측치 계산 및 저장
            missing_count = df[col].isnull().sum()
            missing_ratio = (missing_count / total_rows) * 100
            results_dict[col]['missing'][year_month] = f"{missing_count:,}개 ({missing_ratio:.2f}%)"

            # 데이터 분포 계산 및 저장
            value_counts = df[col].value_counts(dropna=False).sort_index()
            value_ratios = df[col].value_counts(dropna=False, normalize=True).sort_index() * 100

            dist_df = pd.DataFrame({
                '개수': value_counts,
                '비율(%)': value_ratios.round(2)
            })
            results_dict[col]['dist'][year_month] = dist_df

    except Exception as e:
        print(f"{year_month} 데이터 처리 중 오류 발생: {e}")

print("\n분석 완료! 결과를 출력합니다.\n")

# 컬럼별로 결과 묶어서 출력
for col in target_cols:
    kor_name = col_names_kor[col]
    print(f"\n{'='*80}")
    print(f"▶ {col} ({kor_name})")
    print(f"{'='*80}")

    # 결측치 현황 출력
    print("\n[결측치 현황]")
    for year in years:
        if year in results_dict[col]['missing']:
            print(f"- {year}: {results_dict[col]['missing'][year]}")

    # 데이터 분포 현황 병합 및 출력
    print("\n[데이터 분포 현황 (상위 10개)]")
    dist_dfs = []
    valid_years = []
    for year in years:
        if year in results_dict[col]['dist']:
            dist_dfs.append(results_dict[col]['dist'][year])
            valid_years.append(year)

    if dist_dfs:
        # 다중 인덱스(MultiIndex) 컬럼을 생성하여 연도별로 예쁘게 출력
        combined_df = pd.concat(dist_dfs, axis=1, keys=valid_years)
        combined_df.index.name = '값'
        combined_df = combined_df.fillna(0) # 특정 연도에만 없는 값은 0으로 표시
        display(combined_df.head(10))


대출 관련 6개 컬럼 데이터를 분석 중입니다. 잠시만 기다려주세요...

분석 완료! 결과를 출력합니다.


▶ L10210800 (저축은행대출건수)

[결측치 현황]
- 201912: 0개 (0.00%)
- 202012: 0개 (0.00%)
- 202112: 0개 (0.00%)
- 202212: 0개 (0.00%)

[데이터 분포 현황 (상위 10개)]


201912            202012          202112            202212       
          개수  비율(%)         개수  비율(%)       개수  비율(%)         개수  비율(%)
값                                                                      
0  3037720.0  97.08  3035491.0  97.01  3029950  96.83  3019918.0  96.51
1    52059.0   1.66    53541.0   1.71    56619   1.81    62631.0   2.00
2    21034.0   0.67    21328.0   0.68    22650   0.72    24676.0   0.79
3     9515.0   0.30     9899.0   0.32    10415   0.33    11395.0   0.36
4     4536.0   0.14     4592.0   0.15     4928   0.16     5508.0   0.18
5     2099.0   0.07     2179.0   0.07     2302   0.07     2494.0   0.08
6     1059.0   0.03     1038.0   0.03     1124   0.04     1289.0   0.04
7      463.0   0.01      471.0   0.02      507   0.02      552.0   0.02
8      263.0   0.01      253.0   0.01      265   0.01      287.0   0.01
9      131.0   0.00      104.0   0.00      142   0.00      139.0   0.00


▶ L10210B00 (보험대출건수)

[결측치 현황]
- 201912: 0개 (0.00%)
- 202012: 0개 (0.00%)
- 202112: 0개 (0.00%)
- 202212: 0개 (0.00%)

[데이터 분포 현황 (상위 10개)]


201912            202012            202112            202212       
          개수  비율(%)         개수  비율(%)         개수  비율(%)         개수  비율(%)
값                                                                        
0  3049565.0  97.46  3049231.0  97.45  3049037.0  97.44  3047685.0  97.40
1    64712.0   2.07    64747.0   2.07    64981.0   2.08    66074.0   2.11
2    10650.0   0.34    10964.0   0.35    10874.0   0.35    11095.0   0.35
3     2786.0   0.09     2786.0   0.09     2835.0   0.09     2831.0   0.09
4      835.0   0.03      813.0   0.03      777.0   0.02      840.0   0.03
5      241.0   0.01      272.0   0.01      275.0   0.01      265.0   0.01
6      121.0   0.00       95.0   0.00      120.0   0.00      126.0   0.00
7       52.0   0.00       50.0   0.00       53.0   0.00       61.0   0.00
8       36.0   0.00       31.0   0.00       41.0   0.00       24.0   0.00
9       19.0   0.00       26.0   0.00       18.0   0.00       23.0   0.00


▶ L90210300 (할부금융대출건수)

[결측치 현황]
- 201912: 0개 (0.00%)
- 202012: 0개 (0.00%)
- 202112: 0개 (0.00%)
- 202212: 0개 (0.00%)

[데이터 분포 현황 (상위 10개)]


...

201912            202012            202112            202212       
          개수  비율(%)         개수  비율(%)         개수  비율(%)         개수  비율(%)
값                                                                        
0  2903740.0  92.80  2927425.0  93.56  2953968.0  94.41  2963148.0  94.70
1    91705.0   2.93    67034.0   2.14    45392.0   1.45    37703.0   1.20
2    56058.0   1.79    56117.0   1.79    53596.0   1.71    53138.0   1.70
3    30021.0   0.96    30495.0   0.97    29479.0   0.94    29206.0   0.93
4    17005.0   0.54    16996.0   0.54    16822.0   0.54    16413.0   0.52
5     9883.0   0.32    10059.0   0.32     9719.0   0.31     9532.0   0.30
6     6120.0   0.20     6209.0   0.20     5827.0   0.19     5897.0   0.19
7     3835.0   0.12     3983.0   0.13     3786.0   0.12     3806.0   0.12
8     2567.0   0.08     2581.0   0.08     2584.0   0.08     2537.0   0.08
9     1785.0   0.06     1739.0   0.06     1830.0   0.06     1687.0   0.05


▶ L10210M00 (기타업종대출건수)

[결측치 현황]
- 201912: 0개 (0.00%)
- 202012: 0개 (0.00%)
- 202112: 0개 (0.00%)
- 202212: 0개 (0.00%)

[데이터 분포 현황 (상위 10개)]


201912            202012            202112            202212       
          개수  비율(%)         개수  비율(%)         개수  비율(%)         개수  비율(%)
값                                                                        
0  3011035.0  96.23  3009678.0  96.19  3008064.0  96.13  3006373.0  96.08
1    88159.0   2.82    88944.0   2.84    90061.0   2.88    91417.0   2.92
2    20405.0   0.65    20642.0   0.66    21016.0   0.67    21366.0   0.68
3     5995.0   0.19     6203.0   0.20     6346.0   0.20     6284.0   0.20
4     2049.0   0.07     2098.0   0.07     2103.0   0.07     2121.0   0.07
5      736.0   0.02      802.0   0.03      779.0   0.02      749.0   0.02
6      321.0   0.01      317.0   0.01      340.0   0.01      371.0   0.01
7      138.0   0.00      156.0   0.00      140.0   0.00      146.0   0.00
8       92.0   0.00       69.0   0.00       81.0   0.00       75.0   0.00
9       33.0   0.00       43.0   0.00       37.0   0.00       58.0   0.00


▶ L10210000 (총대출건수)

[결측치 현황]
- 201912: 0개 (0.00%)
- 202012: 0개 (0.00%)
- 202112: 0개 (0.00%)
- 202212: 0개 (0.00%)

[데이터 분포 현황 (상위 10개)]


201912            202012            202112            202212       
          개수  비율(%)         개수  비율(%)         개수  비율(%)         개수  비율(%)
값                                                                        
0  1920702.0  61.38  1944033.0  62.13  1970868.0  62.99  1966319.0  62.84
1   569695.0  18.21   572197.0  18.29   569843.0  18.21   576217.0  18.42
2   281522.0   9.00   272074.0   8.70   251552.0   8.04   242685.0   7.76
3   124550.0   3.98   112968.0   3.61   105894.0   3.38   100935.0   3.23
4    56974.0   1.82    42209.0   1.35    35427.0   1.13    31638.0   1.01
5    65927.0   2.11    69111.0   2.21    72929.0   2.33    78813.0   2.52
6    39678.0   1.27    42417.0   1.36    44305.0   1.42    48077.0   1.54
7    24194.0   0.77    25529.0   0.82    27385.0   0.88    29470.0   0.94
8    15161.0   0.48    16001.0   0.51    16763.0   0.54    18150.0   0.58
9     9474.0   0.30    10207.0   0.33    10702.0   0.34    11404.0   0.36

In [ ]:
import pandas as pd

# 분석할 파일 경로 목록
file_paths = [
    '/content/drive/MyDrive/09.개인_CB정보/201912_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202012_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202112_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202212_개인CB.csv'
]

# 분석 대상 3개 컬럼 및 한글명 매핑
target_cols = ['U81203010', 'U81204010', 'U81205010']
col_names_kor = {
    'U81203010': '순자산평가금액_구간1~20',
    'U81204010': '순자산평가금액_구간1~21',
    'U81205010': '자산평가 결과에대한 신뢰수준(A~C)'
}

# 결과를 저장할 딕셔너리 초기화
results_dict = {col: {'missing': {}, 'dist': {}} for col in target_cols}
years = []

print("순자산평가 관련 3개 컬럼 데이터를 분석 중입니다. 잠시만 기다려주세요...")
for path in file_paths:
    year_month = path.split('/')[-1].split('_')[0]
    years.append(year_month)

    try:
        # 메모리 절약을 위해 필요한 컬럼만 로드
        df = pd.read_csv(path, usecols=target_cols)
        total_rows = len(df)

        for col in target_cols:
            # 결측치 계산 및 저장
            missing_count = df[col].isnull().sum()
            missing_ratio = (missing_count / total_rows) * 100
            results_dict[col]['missing'][year_month] = f"{missing_count:,}개 ({missing_ratio:.2f}%)"

            # 데이터 분포 계산 및 저장
            value_counts = df[col].value_counts(dropna=False).sort_index()
            value_ratios = df[col].value_counts(dropna=False, normalize=True).sort_index() * 100

            dist_df = pd.DataFrame({
                '개수': value_counts,
                '비율(%)': value_ratios.round(2)
            })
            results_dict[col]['dist'][year_month] = dist_df

    except Exception as e:
        print(f"{year_month} 데이터 처리 중 오류 발생: {e}")

print("\n분석 완료! 결과를 출력합니다.\n")

# 컬럼별로 결과 묶어서 출력
for col in target_cols:
    kor_name = col_names_kor[col]
    print(f"\n{'='*80}")
    print(f"▶ {col} ({kor_name})")
    print(f"{'='*80}")

    # 결측치 현황 출력
    print("\n[결측치 현황]")
    for year in years:
        if year in results_dict[col]['missing']:
            print(f"- {year}: {results_dict[col]['missing'][year]}")

    # 데이터 분포 현황 병합 및 출력
    if col in ['U81203010', 'U81204010']:
        print("\n[데이터 분포 현황 (전체)]")
    else:
        print("\n[데이터 분포 현황 (상위 10개)]")

    dist_dfs = []
    valid_years = []
    for year in years:
        if year in results_dict[col]['dist']:
            dist_dfs.append(results_dict[col]['dist'][year])
            valid_years.append(year)

    if dist_dfs:
        # 다중 인덱스(MultiIndex) 컬럼을 생성하여 연도별로 예쁘게 출력
        combined_df = pd.concat(dist_dfs, axis=1, keys=valid_years)
        combined_df.index.name = '값'
        combined_df = combined_df.fillna(0) # 특정 연도에만 없는 값은 0으로 표시

        if col in ['U81203010', 'U81204010']:
            display(combined_df)
        else:
            display(combined_df.head(10))


순자산평가 관련 3개 컬럼 데이터를 분석 중입니다. 잠시만 기다려주세요...

분석 완료! 결과를 출력합니다.


▶ U81203010 (순자산평가금액_구간1~20)

[결측치 현황]
- 201912: 0개 (0.00%)
- 202012: 0개 (0.00%)
- 202112: 23개 (0.00%)
- 202212: 20개 (0.00%)

[데이터 분포 현황 (전체)]


201912           202012         202112         202212       
            개수  비율(%)        개수  비율(%)      개수  비율(%)      개수  비율(%)
값                                                                   
0.0     5111.0   0.16    3477.0   0.11    3103   0.10    2925   0.09
1.0    16803.0   0.54   21432.0   0.68   31588   1.01   51770   1.65
2.0    32207.0   1.03   46441.0   1.48   63838   2.04   75093   2.40
3.0    34249.0   1.09   40881.0   1.31   59432   1.90   73558   2.35
4.0    78292.0   2.50   93197.0   2.98  114967   3.67  113565   3.63
5.0    80752.0   2.58   83822.0   2.68   95650   3.06  148280   4.74
6.0    57617.0   1.84   57847.0   1.85   65903   2.11   70162   2.24
7.0    80394.0   2.57   79211.0   2.53   92997   2.97   70290   2.25
8.0   111516.0   3.56   91407.0   2.92  107747   3.44   85715   2.74
9.0   155037.0   4.95  184508.0   5.90  185683   5.93  261102   8.34
10.0   98864.0   3.16   68649.0   2.19   90258   2.88   58311   1.86
11.0  116441.0   3.72  113973.0   3.64   96467   3.08   76810   2.45
12.0  139174.0   4.45  137864.0   4.41  156989   5.02  183880   5.88
13.0  171170.0   5.47  158525.0   5.07  138372   4.42  115688   3.70
14.0  203794.0   6.51  197653.0   6.32  167464   5.35  200286   6.40
15.0  232915.0   7.44  235319.0   7.52  195492   6.25  129163   4.13
16.0  262065.0   8.38  258496.0   8.26  226034   7.22  238835   7.63
17.0  284274.0   9.09  284334.0   9.09  261039   8.34  226078   7.23
18.0  322063.0  10.29  322925.0  10.32  314578  10.05  280223   8.96
19.0  397989.0  12.72  400100.0  12.79  406047  12.98  404960  12.94
20.0  248309.0   7.94  248975.0   7.96  255365   8.16  262322   8.38
NaN        0.0   0.00       0.0   0.00      23   0.00      20   0.00


▶ U81204010 (순자산평가금액_구간1~21)

[결측치 현황]
- 201912: 0개 (0.00%)
- 202012: 9개 (0.00%)
- 202112: 49개 (0.00%)
- 202212: 45개 (0.00%)

[데이터 분포 현황 (전체)]


201912         202012         202112         202212       
            개수  비율(%)      개수  비율(%)      개수  비율(%)      개수  비율(%)
값                                                                 
0.0     5104.0   0.16    3545   0.11    3503   0.11    3064   0.10
1.0    16676.0   0.53   21018   0.67   31073   0.99   50416   1.61
2.0    31466.0   1.01   44360   1.42   60667   1.94   72085   2.30
3.0    33799.0   1.08   37306   1.19   57288   1.83   69728   2.23
4.0    69406.0   2.22   83025   2.65  107746   3.44  110313   3.53
5.0    74161.0   2.37   90838   2.90   95610   3.06  136761   4.37
6.0    54117.0   1.73   50980   1.63   60519   1.93   60803   1.94
7.0    74422.0   2.38   65749   2.10   73012   2.33   68227   2.18
8.0   100682.0   3.22   88403   2.83  115530   3.69   93604   2.99
9.0   132507.0   4.23  169720   5.42  181643   5.81  248221   7.93
10.0   87957.0   2.81   61782   1.97   58993   1.89   45241   1.45
11.0  100974.0   3.23   98676   3.15   75273   2.41   65659   2.10
12.0  121638.0   3.89  121113   3.87  166034   5.31  172329   5.51
13.0  154770.0   4.95  147408   4.71  124462   3.98  102943   3.29
14.0  183497.0   5.86  180046   5.75  169442   5.42  199166   6.37
15.0  217864.0   6.96  219270   7.01  167659   5.36  115482   3.69
16.0  253743.0   8.11  240501   7.69  225984   7.22  225114   7.19
17.0  289237.0   9.24  288061   9.21  262370   8.39  246440   7.88
18.0  336727.0  10.76  334159  10.68  320231  10.23  289044   9.24
19.0  424090.0  13.55  422689  13.51  418128  13.36  405869  12.97
20.0  316875.0  10.13  315554  10.08  315053  10.07  317409  10.14
21.0   49324.0   1.58   44824   1.43   38767   1.24   31073   0.99
NaN        0.0   0.00       9   0.00      49   0.00      45   0.00


▶ U81205010 (자산평가 결과에대한 신뢰수준(A~C))

[결측치 현황]
- 201912: 3,934개 (0.13%)
- 202012: 4,023개 (0.13%)
- 202112: 3,951개 (0.13%)
- 202212: 3,986개 (0.13%)

[데이터 분포 현황 (상위 10개)]


201912          202012          202112          202212       
          개수  비율(%)       개수  비율(%)       개수  비율(%)       개수  비율(%)
값                                                                  
A    1678847  53.65  1679878  53.69  1680199  53.70  1678485  53.64
B     510429  16.31   508687  16.26   509325  16.28   509243  16.27
C     935826  29.91   936448  29.93   935561  29.90   937322  29.96
NaN     3934   0.13     4023   0.13     3951   0.13     3986   0.13

In [ ]:
import pandas as pd

# 분석할 파일 경로 목록
file_paths = [
    '/content/drive/MyDrive/09.개인_CB정보/201912_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202012_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202112_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202212_개인CB.csv'
]

# 분석 대상 컬럼 및 한글명 매핑
target_cols = ['C18210000', 'C1Z001373']
col_names_kor = {
    'C18210000': '체크카드건수',
    'C1Z001373': '3개월내 카드 총이용금액'
}

# 결과를 저장할 딕셔너리 초기화
results_dict = {col: {'missing': {}, 'dist': {}} for col in target_cols}
describe_dict = {} # C1Z001373 describe 결과 저장용
years = []

print("카드 관련 2개 컬럼 데이터를 분석 중입니다. 잠시만 기다려주세요...")
for path in file_paths:
    year_month = path.split('/')[-1].split('_')[0]
    years.append(year_month)

    try:
        # 필요한 컬럼만 로드
        df = pd.read_csv(path, usecols=target_cols)
        total_rows = len(df)

        for col in target_cols:
            # 결측치 계산
            missing_count = df[col].isnull().sum()
            missing_ratio = (missing_count / total_rows) * 100
            results_dict[col]['missing'][year_month] = f"{missing_count:,}개 ({missing_ratio:.2f}%)"

            # 데이터 분포 계산
            value_counts = df[col].value_counts(dropna=False).sort_index()
            value_ratios = df[col].value_counts(dropna=False, normalize=True).sort_index() * 100

            dist_df = pd.DataFrame({
                '개수': value_counts,
                '비율(%)': value_ratios.round(2)
            })
            results_dict[col]['dist'][year_month] = dist_df

        # C1Z001373: 0과 결측치 제외한 describe 계산
        valid_data = df.loc[(df['C1Z001373'].notnull()) & (df['C1Z001373'] != 0), 'C1Z001373']
        describe_dict[year_month] = valid_data.describe().round(2)

    except Exception as e:
        print(f"{year_month} 데이터 처리 중 오류 발생: {e}")

print("\n분석 완료! 결과를 출력합니다.\n")

# 컬럼별로 결과 출력
for col in target_cols:
    kor_name = col_names_kor[col]
    print(f"\n{'='*80}")
    print(f"▶ {col} ({kor_name})")
    print(f"{'='*80}")

    # 결측치 현황
    print("\n[결측치 현황]")
    for year in years:
        if year in results_dict[col]['missing']:
            print(f"- {year}: {results_dict[col]['missing'][year]}")

    # 데이터 분포 현황
    dist_dfs = []
    valid_years = []
    for year in years:
        if year in results_dict[col]['dist']:
            dist_dfs.append(results_dict[col]['dist'][year])
            valid_years.append(year)

    if dist_dfs:
        combined_df = pd.concat(dist_dfs, axis=1, keys=valid_years)
        combined_df.index.name = '값'
        combined_df = combined_df.fillna(0)

        if col == 'C1Z001373':
            print("\n[데이터 분포 현황 (상위 5개 - 값 기준 내림차순)]")
            display(combined_df.sort_index(ascending=False).head(5))

            print("\n[0원 이용 현황]")
            if 0 in combined_df.index or 0.0 in combined_df.index:
                zero_val = 0 if 0 in combined_df.index else 0.0
                display(combined_df.loc[[zero_val]])
            else:
                print("0원 데이터가 없습니다.")

            print("\n[기초 통계량 (0 및 결측치 제외)]")
            # 연도별 describe 결과를 데이터프레임으로 병합
            desc_df = pd.concat(describe_dict.values(), axis=1, keys=describe_dict.keys())
            display(desc_df)
        else:
            print("\n[데이터 분포 현황 (상위 10개)]")
            display(combined_df.head(10))


카드 관련 2개 컬럼 데이터를 분석 중입니다. 잠시만 기다려주세요...

분석 완료! 결과를 출력합니다.


▶ C18210000 (체크카드건수)

[결측치 현황]
- 201912: 0개 (0.00%)
- 202012: 0개 (0.00%)
- 202112: 0개 (0.00%)
- 202212: 0개 (0.00%)

[데이터 분포 현황 (상위 10개)]


201912           202012           202112         202212       
       개수  비율(%)        개수  비율(%)        개수  비율(%)      개수  비율(%)
값                                                                
0  431779  13.80  431224.0  13.78  436193.0  13.94  438758  14.02
1  746122  23.85  732617.0  23.41  726091.0  23.20  722053  23.08
2  730444  23.34  725308.0  23.18  723567.0  23.12  718444  22.96
3  577848  18.47  593665.0  18.97  604316.0  19.31  626510  20.02
4  351878  11.25  355506.0  11.36  361058.0  11.54  361768  11.56
5  171524   5.48  163298.0   5.22  140863.0   4.50  109892   3.51
6   75986   2.43   81303.0   2.60   87076.0   2.78   96623   3.09
7   28275   0.90   30061.0   0.96   32435.0   1.04   35989   1.15
8    9795   0.31   10396.0   0.33   11390.0   0.36   12337   0.39
9    3269   0.10    3533.0   0.11    3781.0   0.12    4150   0.13


▶ C1Z001373 (3개월내 카드 총이용금액)

[결측치 현황]
- 201912: 0개 (0.00%)
- 202012: 0개 (0.00%)
- 202112: 0개 (0.00%)
- 202212: 0개 (0.00%)

[데이터 분포 현황 (상위 5개 - 값 기준 내림차순)]


201912       202012       202112       202212      
           개수 비율(%)     개수 비율(%)     개수 비율(%)     개수 비율(%)
값                                                         
204992    0.0   0.0    0.0   0.0    1.0   0.0    0.0   0.0
200655    0.0   0.0    0.0   0.0    0.0   0.0    1.0   0.0
199842    0.0   0.0    0.0   0.0    0.0   0.0    1.0   0.0
198980    0.0   0.0    0.0   0.0    0.0   0.0    1.0   0.0
198714    0.0   0.0    0.0   0.0    1.0   0.0    0.0   0.0


[0원 이용 현황]


201912           202012           202112           202212       
         개수  비율(%)        개수  비율(%)        개수  비율(%)        개수  비율(%)
값                                                                    
0  423385.0  13.53  391968.0  12.53  331246.0  10.59  331293.0  10.59


[기초 통계량 (0 및 결측치 제외)]


,201912,202012,202112,202212
count,2705651.00,2737068.00,2797790.00,2797743.00
mean,3818.17,4624.63,7885.22,7890.67
std,7846.34,8307.42,11454.35,11171.16
min,-61663.00,-75122.00,-71235.00,-76652.00
25%,14.00,421.00,1135.00,1639.00
50%,2641.00,3035.00,6225.00,5542.00
75%,6351.00,7054.00,12923.00,11374.00
max,196398.00,195879.00,204992.00,200655.00


In [ ]:
import pandas as pd
import glob

# 11번 폴더의 파일 경로 목록 (시간순 정렬)
file_paths_11 = [
    '/content/drive/MyDrive/11.통신카드CB결합/202103_통신카드CB결합.csv',
    '/content/drive/MyDrive/11.통신카드CB결합/202106_통신카드CB결합.csv',
    '/content/drive/MyDrive/11.통신카드CB결합/202109_통신카드CB결합.csv',
    '/content/drive/MyDrive/11.통신카드CB결합/202112_통신카드CB결합.csv',
    '/content/drive/MyDrive/11.통신카드CB결합/202203_통신카드CB결합.csv',
    '/content/drive/MyDrive/11.통신카드CB결합/202206_통신카드CB결합.csv',
    '/content/drive/MyDrive/11.통신카드CB결합/202209_통신카드CB결합.csv',
    '/content/drive/MyDrive/11.통신카드CB결합/202212_통신카드CB결합.csv'
]

print("통신카드CB결합 데이터 현황 확인 중...\n")

for path in file_paths_11:
    year_month = path.split('/')[-1].split('_')[0]
    try:
        # 데이터의 행/열 개수만 확인 (메모리 절약을 위해)
        df_info = pd.read_csv(path, nrows=0) # 컬럼 정보만 로드
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            row_count = sum(1 for _ in f) - 1 # 헤더 제외한 행 개수 계산

        print(f"- {year_month}: {row_count:,} 행, {len(df_info.columns):,} 열")
    except Exception as e:
        print(f"- {year_month} 데이터 로드 중 오류 발생: {e}")

통신카드CB결합 데이터 현황 확인 중...

- 202103: 330,000 행, 738 열
- 202106: 330,000 행, 738 열
- 202109: 330,000 행, 738 열
- 202112: 330,000 행, 738 열
- 202203: 330,000 행, 738 열
- 202206: 330,000 행, 738 열
- 202209: 330,000 행, 738 열
- 202212: 330,000 행, 738 열


In [ ]:
import pandas as pd

# 11번 폴더 씬파일러 판별 대상 컬럼
target_cols_11 = [
    'PYE_C1M210000',
    'PYE_C18233003',
    'PYE_C18233004',
    'PYE_C18233005',
    'PYE_L10210000'
]

results_11 = []

print("11번 폴더 데이터(8개 파일)에서 씬파일러 비중 분석 중...\n")

for path in file_paths_11:
    year_month = path.split('/')[-1].split('_')[0]

    try:
        # 메모리 절약을 위해 지정된 5개 컬럼만 로드
        df_11 = pd.read_csv(path, usecols=target_cols_11)

        # 5개 컬럼의 값이 모두 0인 사람을 씬파일러로 정의
        is_thin_filer = (
            (df_11['PYE_C1M210000'] == 0) &
            (df_11['PYE_C18233003'] == 0) &
            (df_11['PYE_C18233004'] == 0) &
            (df_11['PYE_C18233005'] == 0) &
            (df_11['PYE_L10210000'] == 0)
        )

        thin_filer_count = is_thin_filer.sum()
        total_count = len(df_11)
        ratio = (thin_filer_count / total_count) * 100 if total_count > 0 else 0

        results_11.append({
            '기준년월': year_month,
            '전체 고객 수': total_count,
            '씬파일러 수': thin_filer_count,
            '씬파일러 비중(%)': round(ratio, 2)
        })

    except Exception as e:
        print(f"{year_month} 데이터 처리 중 오류 발생: {e}")

# 결과를 데이터프레임으로 변환하여 출력
if results_11:
    result_df_11 = pd.DataFrame(results_11)
    display(result_df_11)
else:
    print("분석 결과가 없습니다.")

11번 폴더 데이터(8개 파일)에서 씬파일러 비중 분석 중...



,기준년월,전체 고객 수,씬파일러 수,씬파일러 비중(%)
0,202103,330000,44110,13.37
1,202106,330000,44110,13.37
2,202109,330000,44110,13.37
3,202112,330000,44110,13.37
4,202203,330000,41333,12.53
5,202206,330000,41333,12.53
6,202209,330000,41333,12.53
7,202212,330000,41333,12.53


In [ ]:
import pandas as pd

# 분석 대상 컬럼 및 한글명 매핑
target_cols_consume = [
    'R12M_ENT_AMT', 'R12M_HOTEL_AMT', 'R12M_TRAVEL_OS_AMT', 'R12M_DEP_AMT',  # 사치/여가
    'R12M_FOOD_AMT', 'R12M_TRANS_AMT', 'R12M_MED_AMT', 'R12M_MART_AMT', 'R12M_CONV_AMT', # 필수
    'BUY_LUX_YN' # 보조 상호작용
]

col_names_consume_kor = {
    'R12M_ENT_AMT': '유흥',
    'R12M_HOTEL_AMT': '특급호텔',
    'R12M_TRAVEL_OS_AMT': '해외여행',
    'R12M_DEP_AMT': '백화점',
    'R12M_FOOD_AMT': '요식',
    'R12M_TRANS_AMT': '교통',
    'R12M_MED_AMT': '의료',
    'R12M_MART_AMT': '대형할인점',
    'R12M_CONV_AMT': '편의점',
    'BUY_LUX_YN': '명품구매여부'
}

results_consume_dict = {col: {'missing': {}, 'dist': {}} for col in target_cols_consume}
years_11 = []

print("소비 및 명품구매여부 관련 10개 컬럼 데이터를 분석 중입니다. 잠시만 기다려주세요...\n")

for path in file_paths_11:
    year_month = path.split('/')[-1].split('_')[0]
    years_11.append(year_month)

    try:
        # 메모리 절약을 위해 필요한 컬럼만 로드
        df = pd.read_csv(path, usecols=target_cols_consume)
        total_rows = len(df)

        for col in target_cols_consume:
            # 결측치 계산
            missing_count = df[col].isnull().sum()
            missing_ratio = (missing_count / total_rows) * 100
            results_consume_dict[col]['missing'][year_month] = f"{missing_count:,}개 ({missing_ratio:.2f}%)"

            # 데이터 분포 계산 (1% 이상인 값만 필터링)
            all_counts = df[col].value_counts(dropna=False)
            all_ratios = df[col].value_counts(dropna=False, normalize=True) * 100

            mask = all_ratios >= 1.0
            value_counts = all_counts[mask]
            value_ratios = all_ratios[mask]

            dist_df = pd.DataFrame({
                '개수': value_counts,
                '비율(%)': value_ratios.round(2)
            })
            results_consume_dict[col]['dist'][year_month] = dist_df

    except Exception as e:
        print(f"{year_month} 데이터 처리 중 오류 발생: {e}")

print("분석 완료! 결과를 출력합니다.\n")

# 컬럼별로 결과 출력
for col in target_cols_consume:
    kor_name = col_names_consume_kor[col]
    print(f"\n{'='*80}")
    print(f"▶ {col} ({kor_name})")
    print(f"{'='*80}")

    # 결측치 현황
    print("\n[결측치 현황]")
    for year in years_11:
        if year in results_consume_dict[col]['missing']:
            print(f"- {year}: {results_consume_dict[col]['missing'][year]}")

    # 데이터 분포 현황
    print("\n[데이터 분포 현황 (비율 1% 이상)]")
    dist_dfs = []
    valid_years = []
    for year in years_11:
        if year in results_consume_dict[col]['dist']:
            dist_dfs.append(results_consume_dict[col]['dist'][year])
            valid_years.append(year)

    if dist_dfs:
        combined_df = pd.concat(dist_dfs, axis=1, keys=valid_years)
        combined_df.index.name = '값'
        combined_df = combined_df.fillna(0)
        display(combined_df)

소비 및 명품구매여부 관련 10개 컬럼 데이터를 분석 중입니다. 잠시만 기다려주세요...

분석 완료! 결과를 출력합니다.


▶ R12M_ENT_AMT (유흥)

[결측치 현황]
- 202103: 0개 (0.00%)
- 202106: 0개 (0.00%)
- 202109: 0개 (0.00%)
- 202112: 0개 (0.00%)
- 202203: 0개 (0.00%)
- 202206: 0개 (0.00%)
- 202209: 0개 (0.00%)
- 202212: 0개 (0.00%)

[데이터 분포 현황 (비율 1% 이상)]


202103         202106         202109         202112         202203         \
       개수  비율(%)      개수  비율(%)      개수  비율(%)      개수  비율(%)      개수  비율(%)   
값                                                                              
0  329952  99.99  329948  99.98  329938  99.98  329924  99.98  329909  99.97   

   202206         202209         202212         
       개수  비율(%)      개수  비율(%)      개수  비율(%)  
값                                               
0  329841  99.95  329798  99.94  329797  99.94


▶ R12M_HOTEL_AMT (특급호텔)

[결측치 현황]
- 202103: 0개 (0.00%)
- 202106: 0개 (0.00%)
- 202109: 0개 (0.00%)
- 202112: 0개 (0.00%)
- 202203: 0개 (0.00%)
- 202206: 0개 (0.00%)
- 202209: 0개 (0.00%)
- 202212: 0개 (0.00%)

[데이터 분포 현황 (비율 1% 이상)]


202103         202106         202109         202112         202203         \
       개수  비율(%)      개수  비율(%)      개수  비율(%)      개수  비율(%)      개수  비율(%)   
값                                                                              
0  329987  100.0  329979  99.99  329977  99.99  329966  99.99  329965  99.99   

   202206         202209         202212         
       개수  비율(%)      개수  비율(%)      개수  비율(%)  
값                                               
0  329964  99.99  329961  99.99  329962  99.99


▶ R12M_TRAVEL_OS_AMT (해외여행)

[결측치 현황]
- 202103: 0개 (0.00%)
- 202106: 0개 (0.00%)
- 202109: 0개 (0.00%)
- 202112: 0개 (0.00%)
- 202203: 0개 (0.00%)
- 202206: 0개 (0.00%)
- 202209: 0개 (0.00%)
- 202212: 0개 (0.00%)

[데이터 분포 현황 (비율 1% 이상)]


...

202103         202106         202109         202112         202203         \
       개수  비율(%)      개수  비율(%)      개수  비율(%)      개수  비율(%)      개수  비율(%)   
값                                                                              
0  257817  78.13  261190  79.15  261604  79.27  262263  79.47  262979  79.69   

   202206         202209         202212         
       개수  비율(%)      개수  비율(%)      개수  비율(%)  
값                                               
0  264052  80.02  265891  80.57  268468  81.35


▶ R12M_CONV_AMT (편의점)

[결측치 현황]
- 202103: 0개 (0.00%)
- 202106: 0개 (0.00%)
- 202109: 0개 (0.00%)
- 202112: 0개 (0.00%)
- 202203: 0개 (0.00%)
- 202206: 0개 (0.00%)
- 202209: 0개 (0.00%)
- 202212: 0개 (0.00%)

[데이터 분포 현황 (비율 1% 이상)]


202103         202106        202109         202112         202203         \
       개수  비율(%)      개수 비율(%)      개수  비율(%)      개수  비율(%)      개수  비율(%)   
값                                                                             
0  264852  80.26  266979  80.9  267652  81.11  268729  81.43  269009  81.52   

   202206        202209         202212         
       개수 비율(%)      개수  비율(%)      개수  비율(%)  
값                                              
0  269278  81.6  270575  81.99  272706  82.64


▶ BUY_LUX_YN (명품구매여부)

[결측치 현황]
- 202103: 0개 (0.00%)
- 202106: 0개 (0.00%)
- 202109: 0개 (0.00%)
- 202112: 0개 (0.00%)
- 202203: 0개 (0.00%)
- 202206: 0개 (0.00%)
- 202209: 0개 (0.00%)
- 202212: 0개 (0.00%)

[데이터 분포 현황 (비율 1% 이상)]


202103         202106         202109         202112         202203         \
       개수  비율(%)      개수  비율(%)      개수  비율(%)      개수  비율(%)      개수  비율(%)   
값                                                                              
0  330000  100.0  330000  100.0  330000  100.0  330000  100.0  330000  100.0   

   202206         202209         202212         
       개수  비율(%)      개수  비율(%)      개수  비율(%)  
값                                               
0  330000  100.0  330000  100.0  330000  100.0

In [ ]:
import pandas as pd

# 9번 폴더 데이터 경로
file_paths_9 = [
    '/content/drive/MyDrive/09.개인_CB정보/201912_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202012_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202112_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202212_개인CB.csv'
]

# 씬파일러 기준 컬럼 및 분석할 PERF 컬럼
thin_filer_cols = ['C1M210000', 'C18233003', 'C18233004', 'C18233005', 'L10220000']
perf_cols = ['PERF1', 'PERF2', 'PERF3', 'PERF4']
use_cols = thin_filer_cols + perf_cols

results_perf = {col: {} for col in perf_cols}
years_9 = []

print("9번 폴더 씬파일러 대상 PERF1~4 비중 분석 중...\n")

for path in file_paths_9:
    year_month = path.split('/')[-1].split('_')[0]
    years_9.append(year_month)

    try:
        # 메모리 절약을 위해 필요한 컬럼만 로드
        df = pd.read_csv(path, usecols=use_cols)

        # 씬파일러 필터링
        is_thin = (
            (df['C1M210000'] == 0) &
            (df['C18233003'] == 0) &
            (df['C18233004'] == 0) &
            (df['C18233005'] == 0) &
            (df['L10220000'] == 0)
        )

        df_thin = df[is_thin]

        for col in perf_cols:
            # 값 분포 및 비율 계산
            vc = df_thin[col].value_counts(dropna=False).sort_index()
            vr = df_thin[col].value_counts(dropna=False, normalize=True).sort_index() * 100

            dist_df = pd.DataFrame({
                '개수': vc,
                '비율(%)': vr.round(2)
            })
            results_perf[col][year_month] = dist_df

    except Exception as e:
        print(f"{year_month} 데이터 처리 중 오류 발생: {e}")

print("분석 완료! 결과를 출력합니다.\n")

# 컬럼별로 결과 출력
for col in perf_cols:
    print(f"\n{'='*60}")
    print(f"▶ {col}")
    print(f"{'='*60}")

    dist_dfs = []
    valid_years = []
    for year in years_9:
        if year in results_perf[col]:
            dist_dfs.append(results_perf[col][year])
            valid_years.append(year)

    if dist_dfs:
        combined = pd.concat(dist_dfs, axis=1, keys=valid_years)
        combined.index.name = '값'
        combined = combined.fillna(0)
        display(combined)

9번 폴더 씬파일러 대상 PERF1~4 비중 분석 중...

분석 완료! 결과를 출력합니다.


▶ PERF1


201912         202012         202112         202212       
       개수  비율(%)      개수  비율(%)      개수  비율(%)      개수  비율(%)
값                                                            
0  801465  99.82  822278  99.89  832918  99.94  794436  99.96
1    1417   0.18     892   0.11     487   0.06     337   0.04


▶ PERF2


201912         202012         202112         202212       
       개수  비율(%)      개수  비율(%)      개수  비율(%)      개수  비율(%)
값                                                            
0  800397  99.69  821991  99.86  832815  99.93  794359  99.95
1    2485   0.31    1179   0.14     590   0.07     414   0.05


▶ PERF3


201912         202012         202112         202212       
       개수  비율(%)      개수  비율(%)      개수  비율(%)      개수  비율(%)
값                                                            
0  802880  100.0  823167  100.0  833388  100.0  794720  99.99
1       2    0.0       3    0.0      17    0.0      53   0.01


▶ PERF4


201912         202012        202112         202212       
       개수  비율(%)      개수 비율(%)      개수  비율(%)      개수  비율(%)
값                                                           
0  801748  99.86  822314  99.9  832943  99.94  794458  99.96
1    1134   0.14     856   0.1     462   0.06     315   0.04

In [ ]:
import pandas as pd

# 9번 폴더 데이터 경로
file_paths_9 = [
    '/content/drive/MyDrive/09.개인_CB정보/201912_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202012_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202112_개인CB.csv',
    '/content/drive/MyDrive/09.개인_CB정보/202212_개인CB.csv'
]

thin_filer_cols = ['C1M210000', 'C18233003', 'C18233004', 'C18233005', 'L10220000']
perf_cols = ['PERF1', 'PERF2', 'PERF3', 'PERF4']
use_cols = thin_filer_cols + perf_cols

comparison_results = []

print("전체 고객 vs 씬파일러 연체율(PERF) 비교 분석 중...\n")

for path in file_paths_9:
    year_month = path.split('/')[-1].split('_')[0]

    try:
        # 메모리 절약을 위해 필요한 컬럼만 로드
        df = pd.read_csv(path, usecols=use_cols)

        # 씬파일러 필터링
        is_thin = (
            (df['C1M210000'] == 0) &
            (df['C18233003'] == 0) &
            (df['C18233004'] == 0) &
            (df['C18233005'] == 0) &
            (df['L10220000'] == 0)
        )

        df_thin = df[is_thin]

        row_data = {'기준년월': year_month}

        # 각 PERF 컬럼별로 불량(1) 비율 계산 (백분율)
        for col in perf_cols:
            overall_rate = (df[col] == 1).mean() * 100
            thin_rate = (df_thin[col] == 1).mean() * 100
            diff_rate = overall_rate - thin_rate

            row_data[f'전체_{col}(%)'] = round(overall_rate, 3)
            row_data[f'씬파일러_{col}(%)'] = round(thin_rate, 3)
            row_data[f'{col}_차이(%p)'] = round(diff_rate, 3) # %p (퍼센트포인트) 차이

        comparison_results.append(row_data)

    except Exception as e:
        print(f"{year_month} 데이터 처리 중 오류 발생: {e}")

print("분석 완료! 전체 고객과 씬파일러의 연체율 비교 요약표입니다.\n")

# 데이터프레임으로 변환 후 보기 좋게 출력
comp_df = pd.DataFrame(comparison_results)
display(comp_df)


전체 고객 vs 씬파일러 연체율(PERF) 비교 분석 중...

분석 완료! 전체 고객과 씬파일러의 연체율 비교 요약표입니다.



,기준년월,전체_PERF1(%),씬파일러_PERF1(%),PERF1_차이(%p),전체_PERF2(%),씬파일러_PERF2(%),PERF2_차이(%p),전체_PERF3(%),씬파일러_PERF3(%),PERF3_차이(%p),전체_PERF4(%),씬파일러_PERF4(%),PERF4_차이(%p)
0,201912,1.340,0.176,1.164,2.039,0.310,1.730,0.282,0.000,0.282,1.251,0.141,1.110
1,202012,0.801,0.108,0.692,0.983,0.143,0.840,0.170,0.000,0.169,0.789,0.104,0.685
2,202112,0.514,0.058,0.456,0.573,0.071,0.502,0.222,0.002,0.220,0.502,0.055,0.446
3,202212,0.362,0.042,0.320,0.414,0.052,0.362,0.380,0.007,0.373,0.349,0.040,0.309
